# Paper 2 on Google Colab (A100)

Runs the whole study from a fresh Colab session.

**Before running:** `Runtime > Change runtime type > A100 GPU`.

The cells are meant to be run from top to bottom. Every experiment saves its progress,
so after a disconnect you can rerun the same cells and the work continues where it stopped.

## 1. Get the code

The cell clones the repository named in `REPO_URL`.

**The repository is private, so the clone needs a GitHub token.** Create one at `github.com/settings/tokens` with read access to the repository. In Colab, open **Secrets** (the key icon on the left), add it under the name `GITHUB_TOKEN` and switch on *Notebook access*. A public repository needs no token.

Without a repository, leave `REPO_URL` empty: the cell then asks for `dopamine-paper2.zip`, a zip of the project folder. If the code is already in your Drive, set `PROJECT` to that folder instead.

In [ ]:
REPO_URL = 'https://github.com/Hachero98/dopamine-paper2.git'
PROJECT = '/content/dopamine-paper2'

import os
if not os.path.exists(PROJECT):
    if REPO_URL:
        clone_url = REPO_URL
        try:                                     # private repository: use the GITHUB_TOKEN secret
            from google.colab import userdata
            clone_url = REPO_URL.replace('https://', f"https://{userdata.get('GITHUB_TOKEN')}@")
        except Exception:
            pass                                 # no secret: clone without a token
        !git clone -q {clone_url} {PROJECT}
    else:
        from google.colab import files
        archive = next(iter(files.upload()))     # choose dopamine-paper2.zip
        !unzip -q -o "{archive}" -d /content
assert os.path.exists(os.path.join(PROJECT, 'config.py')), \
    f'No config.py in {PROJECT}: check REPO_URL and the GITHUB_TOKEN secret, or upload dopamine-paper2.zip'
%cd {PROJECT}

## 2. Install the pinned JAX stack with CUDA support

The versions are those of Paper 1. Do not upgrade `jax` and `flax` separately.

In [ ]:
!pip install -q "jax[cuda12]==0.10.2" "flax==0.12.5" "optax>=0.2.3" "jaxopt>=0.8"

## 3. Check the device

The output must show a GPU and `float64: True`.

In [ ]:
import jax
import src  # noqa: F401  (importing the package switches float64 on)
from src.utils import io
print('device :', io.device_name())
print('float64:', jax.config.jax_enable_x64)
assert jax.devices()[0].platform == 'gpu', 'No GPU: select an A100 runtime, then restart'
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 4. Keep the results on Google Drive

`results/` becomes a link to a Drive folder. Results then survive the end of the session, and a rerun resumes from them.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_RESULTS = '/content/drive/MyDrive/dopamine-paper2-results'
os.makedirs(DRIVE_RESULTS, exist_ok=True)
if not os.path.islink('results'):
    !cp -rn results/. {DRIVE_RESULTS}/
    !rm -rf results
    os.symlink(DRIVE_RESULTS, 'results')
print('results ->', os.path.realpath('results'))

## 5. Tests

About half a minute. Everything must pass before the experiments are started.

In [ ]:
!python -m pytest -q

## 6. Smoke test of one experiment

`--quick` uses tiny sizes. It only shows that the pipeline runs on this machine; its numbers are not results.

In [ ]:
!python experiments/01_forward_verification.py --quick

## 7. The study

Run in this order. See `docs/EXPERIMENT_GUIDE.md` for what each script answers and how to read its output.

### Stage 1: forward verification
The script stops if the solver is not converged. `--pilot` adds the architecture pilot; choose the network settings from it, set them in `config.py`, then leave them fixed.

In [ ]:
!python experiments/01_forward_verification.py --pilot

### Stages 2 to 7 without the PINN first

These need only the reference solver. They give the identifiability picture of the whole study before any network is trained.

In [ ]:
!python experiments/02_baseline_identifiability.py
!python experiments/04_single_treatment.py --skip-pinn
!python experiments/07_counterfactual_reliability.py

### Inverse problems with the PINN

Time a short pass first, then run the full set. Each realization is saved as it finishes.

In [ ]:
!python experiments/03_baseline_inverse.py --n-realizations 2

In [ ]:
!python experiments/03_baseline_inverse.py
!python experiments/04_single_treatment.py
!python experiments/05_multidose_inverse.py
!python experiments/06_heldout_counterfactual.py

### Robustness

In [ ]:
!python experiments/08_robustness.py --part all

## 8. Look at the output

In [ ]:
from IPython.display import Image, display
from pathlib import Path
for path in sorted(Path('results/figures').glob('*.png')):
    print(path.name)
    display(Image(str(path), width=800))